In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

In [2]:
df = pd.read_csv(r"C:\Users\user\Downloads\comauto_pos.csv")

In [3]:
df.head()

,GRCODE,GRNAME,AccidentYear,DevelopmentYear,DevelopmentLag,IncurLoss_C,CumPaidLoss_C,BulkLoss_C,EarnedPremDIR_C,EarnedPremCeded_C,EarnedPremNet_C,Single,PostedReserve97_C
0,266,Public Underwriters Grp,1988,1988,1,0,0,0,0,0,0,0,932
1,266,Public Underwriters Grp,1988,1989,2,0,0,0,0,0,0,0,932
2,266,Public Underwriters Grp,1988,1990,3,0,0,0,0,0,0,0,932
3,266,Public Underwriters Grp,1988,1991,4,0,0,0,0,0,0,0,932
4,266,Public Underwriters Grp,1988,1992,5,0,0,0,0,0,0,0,932


In [4]:
def build_triangle(df, index_col, column_col, value_col, aggfunc="sum"):
    return df.pivot_table(
        index=index_col,
        columns=column_col,
        values=value_col,
        aggfunc=aggfunc
    )

In [5]:
def incremental_to_cumulative(triangle):
    cumulative = triangle.cumsum(axis=1)
    cumulative.columns = cumulative.columns.astype(int)
    return cumulative

In [6]:
def build_triangle(df, index_col, column_col, value_col, aggfunc="sum"):
    return df.pivot_table(
        index=index_col,
        columns=column_col,
        values=value_col,
        aggfunc=aggfunc
    )

In [7]:
def incremental_to_cumulative(triangle):
    cumulative = triangle.cumsum(axis=1)
    cumulative.columns = cumulative.columns.astype(int)
    return cumulative

In [8]:
def calculate_age_to_age_factors(cumulative_triangle):

    factors = pd.DataFrame(index=cumulative_triangle.index)

    for col in range(cumulative_triangle.shape[1] - 1):
        current = cumulative_triangle.iloc[:, col]
        nxt = cumulative_triangle.iloc[:, col + 1]
        factors[f"{col+1}->{col+2}"] = nxt / current

    return factors

In [9]:
def calculate_average_factors(age_to_age_triangle):

    results = {}

    for col in age_to_age_triangle.columns:

        factors = age_to_age_triangle[col].dropna()

        simple = factors.mean()

        geometric = np.prod(factors) ** (1 / len(factors))

        if len(factors) >= 3:
            medial = (
                factors.sort_values()
                .iloc[1:-1]
                .mean()
            )
        else:
            medial = simple

        results[col] = {
            "Simple": simple,
            "Geometric": geometric,
            "Medial": medial
        }

    return pd.DataFrame(results)

In [10]:
def calculate_cdfs(selected_ldfs):

    cdfs = []

    factors = selected_ldfs.values

    for i in range(len(factors)):
        cdfs.append(np.prod(factors[i:]))

    return pd.Series(cdfs)

In [11]:
def project_ultimates(triangle, cdfs):

    cdfs = cdfs.copy()
    cdfs.index = triangle.columns

    results = []

    for ay in triangle.index:

        row = triangle.loc[ay].dropna()

        latest_age = row.index[-1]
        latest_value = row.iloc[-1]

        cdf = cdfs.loc[latest_age]

        ultimate = latest_value * cdf

        results.append({
            "AccidentYear": ay,
            "LatestAge": latest_age,
            "LatestValue": latest_value,
            "CDF": cdf,
            "Ultimate": ultimate
        })

    return pd.DataFrame(results)

In [12]:
def calculate_unpaid_claim_estimate(ultimate_df):

    reserve_df = ultimate_df.copy()

    reserve_df["TotalReserveEstimate"] = (
        reserve_df["Ultimate"]
        - reserve_df["LatestValue"]
    )

    return reserve_df

In [13]:
def calculate_total_reserve(reserve_df):

    total_reserve = reserve_df["TotalReserveEstimate"].sum()

    return total_reserve

In [14]:
df_company = df[df['GRCODE'] == 13587]

print(df_company.shape)

df_company.head()

(100, 13)


,GRCODE,GRNAME,AccidentYear,DevelopmentYear,DevelopmentLag,IncurLoss_C,CumPaidLoss_C,BulkLoss_C,EarnedPremDIR_C,EarnedPremCeded_C,EarnedPremNet_C,Single,PostedReserve97_C
6500,13587,Chicago Mut Ins Co,1988,1988,1,124,34,3,166,29,137,1,276
6501,13587,Chicago Mut Ins Co,1988,1989,2,122,54,2,166,29,137,1,276
6502,13587,Chicago Mut Ins Co,1988,1990,3,124,78,0,166,29,137,1,276
6503,13587,Chicago Mut Ins Co,1988,1991,4,111,80,0,166,29,137,1,276
6504,13587,Chicago Mut Ins Co,1988,1992,5,91,86,0,166,29,137,1,276


In [15]:
paid = build_triangle(
    df_company,
    index_col='AccidentYear',
    column_col='DevelopmentLag',
    value_col='CumPaidLoss_C',
    aggfunc='sum'
)

paid.head()

DevelopmentLag,1,2,3,4,5,6,7,8,9,10
AccidentYear,,,,,,,,,,
1988,34,54,78,80,86,86,86,86,86,86
1989,14,47,57,64,68,70,76,76,76,76
1990,25,53,58,78,79,128,128,128,128,128
1991,37,59,69,69,69,69,69,69,69,69
1992,11,28,81,113,121,175,205,205,204,204


In [16]:
df_masked = df[df['DevelopmentYear'] <= 1997]

df_masked_company = df_masked[df_masked['GRCODE'] == 13587]

paid_masked = build_triangle(
    df_masked_company,
    index_col='AccidentYear',
    column_col='DevelopmentLag',
    value_col='CumPaidLoss_C',
    aggfunc='sum'
)

paid_masked

DevelopmentLag,1,2,3,4,5,6,7,8,9,10
AccidentYear,,,,,,,,,,
1988,34.0,54.0,78.0,80.0,86.0,86.0,86.0,86.0,86.0,86.0
1989,14.0,47.0,57.0,64.0,68.0,70.0,76.0,76.0,76.0,NaN
1990,25.0,53.0,58.0,78.0,79.0,128.0,128.0,128.0,NaN,NaN
1991,37.0,59.0,69.0,69.0,69.0,69.0,69.0,NaN,NaN,NaN
1992,11.0,28.0,81.0,113.0,121.0,175.0,NaN,NaN,NaN,NaN
1993,10.0,18.0,18.0,22.0,24.0,NaN,NaN,NaN,NaN,NaN
1994,15.0,30.0,119.0,143.0,NaN,NaN,NaN,NaN,NaN,NaN
1995,15.0,25.0,26.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1996,26.0,37.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [17]:
paid_age_to_age = calculate_age_to_age_factors(paid_masked)

paid_age_to_age.head()

,1->2,2->3,3->4,4->5,5->6,6->7,7->8,8->9,9->10
AccidentYear,,,,,,,,,
1988,1.588235,1.444444,1.025641,1.075000,1.000000,1.000000,1.0,1.0,1.0
1989,3.357143,1.212766,1.122807,1.062500,1.029412,1.085714,1.0,1.0,NaN
1990,2.120000,1.094340,1.344828,1.012821,1.620253,1.000000,1.0,NaN,NaN
1991,1.594595,1.169492,1.000000,1.000000,1.000000,1.000000,NaN,NaN,NaN
1992,2.545455,2.892857,1.395062,1.070796,1.446281,NaN,NaN,NaN,NaN


In [18]:
avg_ldfs = calculate_average_factors(paid_age_to_age)

avg_ldfs

,1->2,2->3,3->4,4->5,5->6,6->7,7->8,8->9,9->10
Simple,2.010575,1.727571,1.187463,1.052004,1.219189,1.021429,1.0,1.0,1.0
Geometric,1.941390,1.508088,1.179350,1.051464,1.192573,1.020772,1.0,1.0,1.0
Medial,1.902136,1.475650,1.183436,1.055279,1.158564,1.000000,1.0,1.0,1.0


In [19]:
selected_ldfs = avg_ldfs.loc["Simple"].copy()
selected_ldfs["Tail"] = 1.0

selected_ldfs

1->2     2.010575
2->3     1.727571
3->4     1.187463
4->5     1.052004
5->6     1.219189
6->7     1.021429
7->8     1.000000
8->9     1.000000
9->10    1.000000
Tail     1.000000
Name: Simple, dtype: float64

In [20]:
cdfs = calculate_cdfs(selected_ldfs)

cdfs

0    5.403469
1    2.687525
2    1.555667
3    1.310076
4    1.245315
5    1.021429
6    1.000000
7    1.000000
8    1.000000
9    1.000000
dtype: float64

In [21]:
ultimate_df = project_ultimates(paid_masked, cdfs)
ultimate_df

,AccidentYear,LatestAge,LatestValue,CDF,Ultimate
0,1988,10,86.0,1.000000,86.000000
1,1989,9,76.0,1.000000,76.000000
2,1990,8,128.0,1.000000,128.000000
3,1991,7,69.0,1.000000,69.000000
4,1992,6,175.0,1.021429,178.750000
5,1993,5,24.0,1.245315,29.887552
6,1994,4,143.0,1.310076,187.340931
7,1995,3,26.0,1.555667,40.447346
8,1996,2,37.0,2.687525,99.438423
9,1997,1,25.0,5.403469,135.086731


In [22]:
reserve_df = calculate_unpaid_claim_estimate(ultimate_df)

reserve_df

,AccidentYear,LatestAge,LatestValue,CDF,Ultimate,TotalReserveEstimate
0,1988,10,86.0,1.000000,86.000000,0.000000
1,1989,9,76.0,1.000000,76.000000,0.000000
2,1990,8,128.0,1.000000,128.000000,0.000000
3,1991,7,69.0,1.000000,69.000000,0.000000
4,1992,6,175.0,1.021429,178.750000,3.750000
5,1993,5,24.0,1.245315,29.887552,5.887552
6,1994,4,143.0,1.310076,187.340931,44.340931
7,1995,3,26.0,1.555667,40.447346,14.447346
8,1996,2,37.0,2.687525,99.438423,62.438423
9,1997,1,25.0,5.403469,135.086731,110.086731


In [23]:
total_reserve = calculate_total_reserve(reserve_df)

print(total_reserve)

240.95098367613903


In [25]:
#--------------------------
# chain ladder is over here
#--------------------------

In [26]:
# MACK CHAIN LADDER 


In [ ]:
# Development factor

In [27]:
import numpy as np
import pandas as pd

triangle = paid_masked.copy()

ldfs = []

for j in range(triangle.shape[1]-1):

    current = triangle.iloc[:, j]
    nxt = triangle.iloc[:, j+1]

    mask = current.notna() & nxt.notna()

    f_j = nxt[mask].sum() / current[mask].sum()

    ldfs.append(f_j)

ldfs = pd.Series(
    ldfs,
    index=[f"{triangle.columns[i]}->{triangle.columns[i+1]}"
           for i in range(len(ldfs))]
)

print(ldfs)

1->2     1.877005
2->3     1.611465
3->4     1.185417
4->5     1.049296
5->6     1.248227
6->7     1.016997
7->8     1.000000
8->9     1.000000
9->10    1.000000
dtype: float64


In [28]:
# MACK Sigma Squared

In [29]:
sigma2 = []

for j in range(triangle.shape[1]-2):

    current = triangle.iloc[:, j]
    nxt = triangle.iloc[:, j+1]

    mask = current.notna() & nxt.notna()

    x = current[mask]
    y = nxt[mask]

    f = y.sum() / x.sum()

    n = len(x)

    s2 = ((x * ((y/x) - f)**2).sum()) / (n - 1)

    sigma2.append(s2)

sigma2 = pd.Series(
    sigma2,
    index=[f"{triangle.columns[i]}->{triangle.columns[i+1]}"
           for i in range(len(sigma2))]
)

print(sigma2)

1->2     6.144483
2->3    37.422061
3->4     1.612767
4->5     0.085160
5->6     7.121643
6->7     0.137434
7->8     0.000000
8->9     0.000000
dtype: float64


In [30]:
# Sigma

In [31]:
sigma = np.sqrt(sigma2)

print(sigma)

1->2    2.478807
2->3    6.117357
3->4    1.269947
4->5    0.291822
5->6    2.668641
6->7    0.370721
7->8    0.000000
8->9    0.000000
dtype: float64


In [32]:
# Number of observations

In [33]:
n_obs = []

for j in range(triangle.shape[1]-1):

    current = triangle.iloc[:, j]
    nxt = triangle.iloc[:, j+1]

    mask = current.notna() & nxt.notna()

    n_obs.append(mask.sum())

n_obs = pd.Series(
    n_obs,
    index=ldfs.index
)

print(n_obs)

1->2     9
2->3     8
3->4     7
4->5     6
5->6     5
6->7     4
7->8     3
8->9     2
9->10    1
dtype: int64


In [34]:
# MACK variance 

In [35]:
mack_results = []

for ay in triangle.index:

    row = triangle.loc[ay]
    observed = row.dropna()

    latest_age = observed.index[-1]
    latest_value = observed.iloc[-1]

    reserve = reserve_df.loc[
        reserve_df["AccidentYear"] == ay,
        "TotalReserveEstimate"
    ].iloc[0]

    variance_factor = 0

    age_position = list(triangle.columns).index(latest_age)

    for j in range(age_position, len(sigma2)):

        variance_factor += (
            sigma2.iloc[j]
            /
            (ldfs.iloc[j] ** 2)
        )

    mack_variance = (
        reserve ** 2
        *
        variance_factor
        /
        latest_value
    )

    mack_results.append({
        "AccidentYear": ay,
        "Reserve": reserve,
        "MackVariance": mack_variance
    })

mack_results = pd.DataFrame(mack_results)

mack_results

,AccidentYear,Reserve,MackVariance
0,1988,0.000000,0.000000
1,1989,0.000000,0.000000
2,1990,0.000000,0.000000
3,1991,0.000000,0.000000
4,1992,3.750000,0.010678
5,1993,5.887552,6.793550
6,1994,44.340931,65.734811
7,1995,14.447346,47.595401
8,1996,62.438423,2143.096314
9,1997,110.086731,10705.274622


In [36]:
# MACK Standard Error

In [37]:
mack_results["MackStdError"] = np.sqrt(
    mack_results["MackVariance"]
)

mack_results

,AccidentYear,Reserve,MackVariance,MackStdError
0,1988,0.000000,0.000000,0.000000
1,1989,0.000000,0.000000,0.000000
2,1990,0.000000,0.000000,0.000000
3,1991,0.000000,0.000000,0.000000
4,1992,3.750000,0.010678,0.103333
5,1993,5.887552,6.793550,2.606444
6,1994,44.340931,65.734811,8.107701
7,1995,14.447346,47.595401,6.898942
8,1996,62.438423,2143.096314,46.293588
9,1997,110.086731,10705.274622,103.466297


In [38]:
# Coefficient of Variation

In [39]:
mack_results["CV"] = np.where(
    mack_results["Reserve"] > 0,
    mack_results["MackStdError"] / mack_results["Reserve"],
    0
)

mack_results

,AccidentYear,Reserve,MackVariance,MackStdError,CV
0,1988,0.000000,0.000000,0.000000,0.000000
1,1989,0.000000,0.000000,0.000000,0.000000
2,1990,0.000000,0.000000,0.000000,0.000000
3,1991,0.000000,0.000000,0.000000,0.000000
4,1992,3.750000,0.010678,0.103333,0.027556
5,1993,5.887552,6.793550,2.606444,0.442704
6,1994,44.340931,65.734811,8.107701,0.182849
7,1995,14.447346,47.595401,6.898942,0.477523
8,1996,62.438423,2143.096314,46.293588,0.741428
9,1997,110.086731,10705.274622,103.466297,0.939862


In [40]:
# Portfolio Level 

In [41]:
total_reserve = mack_results["Reserve"].sum()

total_mack_variance = mack_results["MackVariance"].sum()

total_mack_std_error = np.sqrt(total_mack_variance)

total_cv = total_mack_std_error / total_reserve

print("Total Reserve =", total_reserve)
print("Total Mack Variance =", total_mack_variance)
print("Total Mack Std Error =", total_mack_std_error)
print("Total CV =", total_cv)

Total Reserve = 240.95098367613903
Total Mack Variance = 12968.505376310151
Total Mack Std Error = 113.87934569670723
Total CV = 0.4726245311775605


In [42]:
# =====================================================
# Mack Chain Ladder is Over
# =====================================================